## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/insurance_fraud/processed/'
MODEL_DIR  = '/content/drive/MyDrive/insurance_fraud/models/'
REPORT_DIR = '/content/drive/MyDrive/insurance_fraud/reports/'
SCAN_DIR   = '/content/drive/MyDrive/insurance_fraud/scanned/'
API_DIR    = '/content/insurance_fraud_api/'
os.makedirs(API_DIR, exist_ok=True)
os.makedirs(SCAN_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print('✅ Drive mounted.')

## Cell 2 — Install All Dependencies

In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio \
             easyocr pdf2image pillow reportlab \
             xgboost lightgbm shap gradio \
             python-multipart aiofiles --quiet
!apt-get install -y poppler-utils --quiet

import pandas as pd
import numpy as np
import json
import joblib
import shap
import re
import io
import os
import time
import warnings
import asyncio
import nest_asyncio
import threading
import uvicorn
import requests
import gradio as gr
from datetime import datetime
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
nest_asyncio.apply()   # Allows uvicorn inside Colab's event loop

# FastAPI
from fastapi import FastAPI, File, UploadFile, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, FileResponse
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any

# ReportLab for PDF
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, HRFlowable
)
from reportlab.lib.enums import TA_CENTER

NAVY='#0D1B3E'; ACCENT='#1DB8C4'; GREEN='#22C97A'
RED='#E84C4C'; YELLOW='#F5C842'; WHITE='#FFFFFF'; GRAY='#8BA3BF'

print('✅ All packages installed and imported.')

## Cell 3 — Load All Models & Artifacts

In [ ]:
print('📦 Loading all models and artifacts...')

xgb_model  = joblib.load(MODEL_DIR + 'model_xgboost.pkl')
lgbm_model = joblib.load(MODEL_DIR + 'model_lightgbm.pkl')
rf_model   = joblib.load(MODEL_DIR + 'model_random_forest.pkl')
scaler     = joblib.load(MODEL_DIR + 'scaler.pkl')

with open(OUTPUT_DIR + 'final_feature_columns.json') as f:
    feature_cols = json.load(f)
with open(MODEL_DIR + 'model_metadata.json') as f:
    metadata = json.load(f)
with open(MODEL_DIR + 'ensemble_config.json') as f:
    ens_config = json.load(f)

# Population stats for comparison
X_val    = pd.read_parquet(MODEL_DIR + 'X_val.parquet')
X_val    = X_val[[c for c in feature_cols if c in X_val.columns]]
pop_means = X_val.mean().to_dict()
pop_stds  = X_val.std().to_dict()

# SHAP explainer
explainer = shap.TreeExplainer(xgb_model)

THRESHOLD = metadata['best_threshold']
WEIGHTS   = ens_config['weights']   # [3, 3, 1]
W_TOTAL   = sum(WEIGHTS)

print(f'✅ Models loaded:')
print(f'   XGBoost, LightGBM, RandomForest')
print(f'   Feature columns : {len(feature_cols)}')
print(f'   Threshold       : {THRESHOLD}')
print(f'   Model AUC-ROC   : {metadata["val_auc_roc"]}')

## Cell 4 — Core Inference Functions

In [ ]:
def ensemble_predict(feat_df):
    """
    Runs the weighted ensemble (XGB×3 + LGBM×3 + RF×1)
    Returns fraud_score (float) and individual model probabilities.
    """
    feat_df = feat_df.copy()
    feat_df = feat_df.fillna(0).replace([np.inf, -np.inf], 0)

    p_xgb  = xgb_model.predict_proba(feat_df)[:, 1]
    p_lgbm = lgbm_model.predict_proba(feat_df)[:, 1]
    p_rf   = rf_model.predict_proba(feat_df)[:, 1]

    p_ens = (WEIGHTS[0]*p_xgb + WEIGHTS[1]*p_lgbm + WEIGHTS[2]*p_rf) / W_TOTAL
    return {
        'fraud_score' : float(round(p_ens[0], 4)),
        'xgb_score'   : float(round(p_xgb[0],  4)),
        'lgbm_score'  : float(round(p_lgbm[0], 4)),
        'rf_score'    : float(round(p_rf[0],   4)),
    }


def get_shap_reasons(feat_df, top_n=5):
    """Returns top N SHAP-based natural language fraud reasons."""
    sv = explainer(feat_df)
    shap_series = pd.Series(sv.values[0], index=feature_cols)
    top = shap_series[shap_series > 0].sort_values(ascending=False).head(top_n)

    TEMPLATES = {
        'ip_avg_reimbursement'     : 'Avg claim reimbursement is {ratio:.1f}x population average',
        'ip_total_reimbursement'   : 'Total billing {ratio:.1f}x higher than typical provider',
        'ip_claim_count'           : 'Filed {val:.0f} claims — {ratio:.1f}x more than average',
        'total_unique_patients'    : 'Served {val:.0f} unique patients — {ratio:.1f}x typical',
        'ip_avg_stay_days'         : 'Avg hospital stay {val:.1f} days — {ratio:.1f}x longer than usual',
        'ip_avg_chronic_cond'      : 'Patients avg {val:.2f} chronic conditions — possible diagnosis stuffing',
        'ip_unique_attending_phys' : 'Used {val:.0f} unique physicians — unusual pattern',
        'feat_reimb_per_unique_patient':'Extracts ${val:,.0f} per patient — {ratio:.1f}x average',
        'risk_composite_score'     : 'Overall fraud risk score: {val:.3f} (high)',
        'risk_financial'           : 'Financial anomaly score: {val:.3f}',
        'feat_total_flags_triggered': 'Triggered {val:.0f}/8 extreme outlier flags',
        'interact_volume_x_reimb'  : 'High volume × high reimbursement — billing mill pattern',
        'interact_stay_x_reimb'    : 'Long stay × high claim — overbilling signal',
        'ip_deceased_patient_count': 'Claims filed for {val:.0f} deceased patients',
    }

    reasons = []
    for feat, sv_val in top.items():
        val = float(feat_df[feat].iloc[0])
        avg = pop_means.get(feat, 1e-9)
        ratio = val / max(avg, 1e-9)
        tmpl = TEMPLATES.get(feat,
            f'{feat.replace("_"," ").title()}: {{val:.3f}} ({{ratio:.1f}}x avg)')
        try:
            txt = tmpl.format(val=val, avg=avg, ratio=ratio)
        except:
            txt = feat
        reasons.append({'feature': feat, 'shap': round(float(sv_val), 4), 'text': txt})
    return reasons


def get_risk_tier(score):
    if score >= 0.85:      return 'CRITICAL', 'Immediate investigation — do not process'
    elif score >= THRESHOLD: return 'HIGH',   'Flag for detailed manual review'
    elif score >= THRESHOLD*0.7: return 'MEDIUM','Process with additional verification'
    else:                  return 'LOW',      'Approve — within normal parameters'


def feat_df_from_dict(feature_dict):
    """Builds a feature DataFrame from a dict — fills missing cols with pop means."""
    row = {col: pop_means.get(col, 0.0) for col in feature_cols}
    for k, v in feature_dict.items():
        if k in row:
            row[k] = float(v)
    df = pd.DataFrame([row])[feature_cols]
    return df.fillna(0).replace([np.inf, -np.inf], 0)


print('✅ Core inference functions defined.')

## Cell 5 — OCR & Document Scan Functions (from M6)

In [ ]:
# Initialize EasyOCR once
print('🔄 Initializing EasyOCR...')
import easyocr
ocr_reader = easyocr.Reader(['en'], gpu=True)
print('✅ EasyOCR ready.')

FIELD_PATTERNS = {
    'claim_amount'        : [r'(?:claim\s*amount|total\s*claim)[\s:]*[\$₹Rs\.]*([\d,]+(?:\.\d{1,2})?)'],
    'reimbursement_amount': [r'(?:reimburs)[\w\s]*[:\s]*[\$₹Rs\.]*([\d,]+(?:\.\d{1,2})?)'],
    'hospital_days'       : [r'([\d]+)\s*(?:days?|nights?)\s*(?:of\s*)?(?:stay|admission)'],
    'patient_age'         : [r'(?:age|aged?)[\s:]*([\d]{1,3})\s*(?:years?|yrs?)?'],
    'num_diagnoses'       : [r'([\d]+)\s*(?:diagnos|conditions?|chronic)'],
    'deductible_amount'   : [r'(?:deductible|deduct)[\s:]*[\$₹Rs\.]*([\d,]+(?:\.\d{1,2})?)'],
    'doctor_id'           : [r'(?:physician|doctor|attending)[\s:]*(?:id)?[\s:]*([A-Z]{2,4}\d{4,})'],
    'patient_id'          : [r'(?:patient|beneficiary)[\s:]*(?:id)[\s:]*([A-Z0-9]{6,})'],
    'diagnosis_code'      : [r'([A-Z]\d{2}\.?\d{0,2})(?:\s|,)'],
    'admission_date'      : [r'(?:admit|admission)[\s:]*([\d]{1,2}[/-][\d]{1,2}[/-][\d]{2,4})'],
    'discharge_date'      : [r'(?:discharge)[\s:]*([\d]{1,2}[/-][\d]{1,2}[/-][\d]{2,4})'],
}

def parse_fields(raw_text):
    extracted = {}
    for field, patterns in FIELD_PATTERNS.items():
        for pat in patterns:
            m = re.search(pat, raw_text, re.IGNORECASE)
            if m:
                v = m.group(1).strip()
                if field in ['claim_amount','reimbursement_amount','deductible_amount',
                              'hospital_days','patient_age','num_diagnoses']:
                    try:
                        extracted[field] = float(re.sub(r'[,\s]','', v))
                    except:
                        extracted[field] = v
                else:
                    extracted[field] = v
                break
    # Compute stay days from dates
    if 'admission_date' in extracted and 'discharge_date' in extracted:
        try:
            from dateutil import parser as dp
            a = dp.parse(str(extracted['admission_date']), dayfirst=True)
            d = dp.parse(str(extracted['discharge_date']), dayfirst=True)
            extracted['hospital_days'] = float(max(0,(d-a).days))
        except: pass
    return extracted


def fields_to_feat_df(fields):
    """Map claim fields to model feature vector."""
    row = {col: pop_means.get(col, 0.0) for col in feature_cols}
    eps = 1e-9
    if 'claim_amount' in fields and isinstance(fields['claim_amount'], float):
        a = fields['claim_amount']
        for c in ['ip_avg_reimbursement','ip_total_reimbursement']:
            if c in row: row[c] = a
    if 'hospital_days' in fields and isinstance(fields['hospital_days'], float):
        d = min(fields['hospital_days'], 365)
        for c in ['ip_avg_stay_days','ip_max_stay_days','ip_total_stay_days']:
            if c in row: row[c] = d
    if 'patient_age' in fields and isinstance(fields['patient_age'], float):
        age = min(fields['patient_age'], 120)
        for c in ['ip_avg_patient_age','op_avg_patient_age']:
            if c in row: row[c] = age
    if 'num_diagnoses' in fields and isinstance(fields['num_diagnoses'], float):
        n = fields['num_diagnoses']
        for c in ['ip_avg_chronic_cond','ip_max_chronic_cond']:
            if c in row: row[c] = min(n, 11)
    if 'deductible_amount' in fields and isinstance(fields['deductible_amount'], float):
        ded = fields['deductible_amount']
        for c in ['ip_total_deductible','ip_avg_deductible']:
            if c in row: row[c] = ded
    # Derived
    reimb = row.get('ip_avg_reimbursement', 0)
    stays = row.get('ip_avg_stay_days', 0)
    ded   = row.get('ip_total_deductible', 0)
    if 'feat_reimb_per_ip_claim'       in row: row['feat_reimb_per_ip_claim']       = reimb
    if 'feat_reimb_per_unique_patient' in row: row['feat_reimb_per_unique_patient'] = reimb
    if 'feat_deductible_ratio'         in row: row['feat_deductible_ratio']         = ded/(reimb+eps)
    if 'feat_stay_days_per_claim'      in row: row['feat_stay_days_per_claim']      = stays
    if 'risk_financial'    in row: row['risk_financial']    = min(reimb/max(pop_means.get('ip_avg_reimbursement',1),eps)/3, 1)
    if 'risk_stay_duration'in row: row['risk_stay_duration']= min(stays/max(pop_means.get('ip_avg_stay_days',1)*3,eps), 1)
    if 'risk_composite_score' in row:
        row['risk_composite_score'] = np.mean([row.get('risk_financial',0.5), row.get('risk_stay_duration',0.5)])
    if 'interact_stay_x_reimb' in row:
        row['interact_stay_x_reimb'] = stays * np.log1p(reimb)
    if 'interact_composite_squared' in row:
        row['interact_composite_squared'] = row.get('risk_composite_score',0.5)**2
    for c in feature_cols:
        if c.startswith('zscore_'):
            base = c.replace('zscore_','')
            if base in row:
                row[c] = (row[base]-pop_means.get(base,0)) / max(pop_stds.get(base,1),eps)
    df = pd.DataFrame([row])[feature_cols]
    return df.fillna(0).replace([np.inf,-np.inf],0)


def scan_bytes(file_bytes, filename):
    """Full scan pipeline for in-memory file bytes (used by API)."""
    ext = Path(filename).suffix.lower()
    tmp = f'/tmp/upload_{datetime.now().strftime("%H%M%S")}{ext}'
    with open(tmp,'wb') as f: f.write(file_bytes)

    if ext == '.pdf':
        from pdf2image import convert_from_path
        pil_img = convert_from_path(tmp, dpi=200)[0]
    else:
        pil_img = Image.open(tmp).convert('RGB')

    ocr_results = ocr_reader.readtext(np.array(pil_img))
    raw_text    = ' '.join([r[1] for r in ocr_results])
    fields      = parse_fields(raw_text)
    feat_df     = fields_to_feat_df(fields)
    preds       = ensemble_predict(feat_df)
    reasons     = get_shap_reasons(feat_df)
    tier, action= get_risk_tier(preds['fraud_score'])

    return {
        'fraud_score'    : preds['fraud_score'],
        'xgb_score'      : preds['xgb_score'],
        'lgbm_score'     : preds['lgbm_score'],
        'rf_score'       : preds['rf_score'],
        'fraud_predicted': preds['fraud_score'] >= THRESHOLD,
        'risk_tier'      : tier,
        'action'         : action,
        'extracted_fields': {k: str(v) for k,v in fields.items()},
        'top_reasons'    : reasons,
        'ocr_regions'    : len(ocr_results),
        'scanned_at'     : datetime.now().isoformat(),
    }, pil_img, ocr_results


print('✅ OCR + scan pipeline ready.')

## Cell 6 — PDF Report Generator

In [ ]:
def generate_report_pdf(result, output_path):
    """Generates a single-provider fraud investigation PDF."""
    doc    = SimpleDocTemplate(output_path, pagesize=A4,
                               leftMargin=2*cm, rightMargin=2*cm,
                               topMargin=2*cm, bottomMargin=2*cm)
    styles = getSampleStyleSheet()
    story  = []

    ts = ParagraphStyle('TS', parent=styles['Title'],
        fontSize=18, textColor=colors.HexColor('#1DB8C4'), alignment=TA_CENTER)
    ss = ParagraphStyle('SS', parent=styles['Normal'],
        fontSize=10, textColor=colors.HexColor('#888888'), alignment=TA_CENTER)
    hs = ParagraphStyle('HS', parent=styles['Heading2'],
        fontSize=13, textColor=colors.HexColor('#1A3A6B'))
    bs = ParagraphStyle('BS', parent=styles['Normal'], fontSize=10)

    tier_hex = {'CRITICAL':'#E84C4C','HIGH':'#FF8C42','MEDIUM':'#F5C842','LOW':'#22C97A'}
    tc = tier_hex.get(result.get('risk_tier','LOW'), '#333333')

    story.append(Paragraph('FRAUD INVESTIGATION REPORT', ts))
    story.append(Paragraph(
        f'Generated: {datetime.now().strftime("%B %d, %Y %H:%M")} | '
        f'Model AUC: {metadata["val_auc_roc"]}', ss))
    story.append(HRFlowable(width='100%', thickness=1.5,
                             color=colors.HexColor('#1DB8C4')))
    story.append(Spacer(1, 0.4*cm))

    # Score summary
    score_data = [
        ['Field', 'Value'],
        ['Fraud Score',   f"{result['fraud_score']:.4f}"],
        ['Risk Tier',     result.get('risk_tier','N/A')],
        ['Decision',      'FRAUD' if result.get('fraud_predicted') else 'LEGITIMATE'],
        ['Action',        result.get('action','N/A')],
        ['XGBoost Score', f"{result.get('xgb_score',0):.4f}"],
        ['LightGBM Score',f"{result.get('lgbm_score',0):.4f}"],
        ['Scanned At',    result.get('scanned_at', datetime.now().isoformat())[:19]],
    ]
    t = Table(score_data, colWidths=[8*cm, 8*cm])
    t.setStyle(TableStyle([
        ('BACKGROUND',(0,0),(-1,0),colors.HexColor('#1A3A6B')),
        ('TEXTCOLOR', (0,0),(-1,0),colors.white),
        ('FONTNAME',  (0,0),(-1,0),'Helvetica-Bold'),
        ('FONTSIZE',  (0,0),(-1,-1),10),
        ('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.HexColor('#F5F8FF'), colors.white]),
        ('GRID',      (0,0),(-1,-1),0.5,colors.grey),
        ('PADDING',   (0,0),(-1,-1),8),
    ]))
    story.append(t)
    story.append(Spacer(1, 0.5*cm))

    # Fraud reasons
    story.append(Paragraph('Top Fraud Signals', hs))
    for i, r in enumerate(result.get('top_reasons',[])[:5], 1):
        story.append(Paragraph(
            f"{i}. {r['text']}  [SHAP: +{r['shap']:.4f}]", bs))
        story.append(Spacer(1, 0.1*cm))

    story.append(Spacer(1, 0.4*cm))

    # Extracted fields
    if result.get('extracted_fields'):
        story.append(Paragraph('Extracted Claim Fields (OCR)', hs))
        ef = result['extracted_fields']
        ef_data = [['Field','Extracted Value']] + [[k,str(v)] for k,v in ef.items()]
        et = Table(ef_data, colWidths=[8*cm, 8*cm])
        et.setStyle(TableStyle([
            ('BACKGROUND',(0,0),(-1,0),colors.HexColor('#1A3A6B')),
            ('TEXTCOLOR', (0,0),(-1,0),colors.white),
            ('FONTNAME',  (0,0),(-1,0),'Helvetica-Bold'),
            ('FONTSIZE',  (0,0),(-1,-1),9),
            ('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.HexColor('#F5F8FF'), colors.white]),
            ('GRID',      (0,0),(-1,-1),0.5,colors.grey),
            ('PADDING',   (0,0),(-1,-1),6),
        ]))
        story.append(et)

    doc.build(story)
    return output_path


print('✅ PDF report generator defined.')

## Cell 7 — FastAPI Application

In [ ]:
# ── Pydantic request/response models ──────────────────────────────────────────
class PredictRequest(BaseModel):
    features: Dict[str, float] = Field(
        ...,
        description='Dict of feature_name → value. Missing features default to population mean.',
        example={'ip_avg_reimbursement': 15000.0, 'ip_avg_stay_days': 12.5, 'ip_claim_count': 45.0}
    )
    explain: bool = Field(True, description='Whether to return SHAP reasons')


class PredictResponse(BaseModel):
    fraud_score    : float
    fraud_predicted: bool
    risk_tier      : str
    action         : str
    xgb_score      : float
    lgbm_score     : float
    rf_score       : float
    top_reasons    : List[Dict[str, Any]]
    threshold_used : float
    model_auc      : float
    predicted_at   : str


# ── Build FastAPI app ──────────────────────────────────────────────────────────
app = FastAPI(
    title       = '🛡️ Insurance Fraud Detection API',
    description = (
        'Multimodal AI fraud detection system. '
        'Accepts tabular provider features OR raw claim documents (PDF/image). '
        'Returns fraud probability + SHAP explanations + risk tier.'
    ),
    version     = '1.0.0',
    docs_url    = '/docs',
    redoc_url   = '/redoc',
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


# ── Endpoints ──────────────────────────────────────────────────────────────────
@app.get('/health', tags=['System'])
async def health_check():
    """API health check — returns model status and version info."""
    return {
        'status'          : 'healthy',
        'model'           : 'Ensemble (XGB + LGBM + RF)',
        'model_auc'       : metadata['val_auc_roc'],
        'threshold'       : THRESHOLD,
        'feature_count'   : len(feature_cols),
        'version'         : '1.0.0',
        'timestamp'       : datetime.now().isoformat(),
    }


@app.post('/predict', response_model=PredictResponse, tags=['Fraud Detection'])
async def predict_fraud(request: PredictRequest):
    """
    **Predict fraud from provider features.**

    Accepts a dict of feature values. Any missing features default to
    population mean. Returns ensemble fraud score + SHAP reasons.

    Example features: `ip_avg_reimbursement`, `ip_avg_stay_days`,
    `ip_claim_count`, `total_unique_patients`, `ip_avg_chronic_cond`
    """
    try:
        feat_df = feat_df_from_dict(request.features)
        preds   = ensemble_predict(feat_df)
        tier, action = get_risk_tier(preds['fraud_score'])
        reasons = get_shap_reasons(feat_df) if request.explain else []

        return PredictResponse(
            fraud_score     = preds['fraud_score'],
            fraud_predicted = preds['fraud_score'] >= THRESHOLD,
            risk_tier       = tier,
            action          = action,
            xgb_score       = preds['xgb_score'],
            lgbm_score      = preds['lgbm_score'],
            rf_score        = preds['rf_score'],
            top_reasons     = reasons,
            threshold_used  = THRESHOLD,
            model_auc       = metadata['val_auc_roc'],
            predicted_at    = datetime.now().isoformat(),
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post('/scan-document', tags=['Document Scanner'])
async def scan_document_endpoint(
    file: UploadFile = File(..., description='Insurance claim document — PDF, JPG, or PNG'),
    generate_pdf: bool = False
):
    """
    **Scan an uploaded insurance claim document.**

    Accepts PDF or image. Runs OCR → field extraction → fraud scoring → SHAP.
    Returns fraud score, per-field suspicion, and optional PDF report.
    """
    allowed = {'.pdf', '.jpg', '.jpeg', '.png', '.bmp'}
    ext     = Path(file.filename).suffix.lower()
    if ext not in allowed:
        raise HTTPException(400, f'Unsupported file type: {ext}. Use: {allowed}')

    try:
        contents = await file.read()
        result, pil_img, _ = scan_bytes(contents, file.filename)

        response = {
            'filename'       : file.filename,
            'file_size_kb'   : round(len(contents)/1024, 1),
            **result
        }

        if generate_pdf:
            ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
            pdf = f'/tmp/report_{ts}.pdf'
            generate_report_pdf(result, pdf)
            response['pdf_report'] = pdf

        return JSONResponse(content=response)

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post('/batch-predict', tags=['Fraud Detection'])
async def batch_predict(requests_list: List[PredictRequest]):
    """
    **Score multiple providers in one API call.**

    Accepts a list of feature dicts. Returns a list of fraud scores.
    Max 500 providers per call.
    """
    if len(requests_list) > 500:
        raise HTTPException(400, 'Max 500 providers per batch call')

    results = []
    for req in requests_list:
        try:
            feat_df = feat_df_from_dict(req.features)
            preds   = ensemble_predict(feat_df)
            tier, action = get_risk_tier(preds['fraud_score'])
            results.append({
                'fraud_score'    : preds['fraud_score'],
                'fraud_predicted': preds['fraud_score'] >= THRESHOLD,
                'risk_tier'      : tier,
                'action'         : action,
            })
        except Exception as e:
            results.append({'error': str(e)})

    return {
        'total'         : len(results),
        'fraud_count'   : sum(1 for r in results if r.get('fraud_predicted')),
        'results'       : results,
        'processed_at'  : datetime.now().isoformat(),
    }


@app.get('/features', tags=['System'])
async def list_features():
    """Returns all available feature names with population statistics."""
    feat_info = []
    for col in feature_cols:
        feat_info.append({
            'name'   : col,
            'mean'   : round(pop_means.get(col, 0), 4),
            'std'    : round(pop_stds.get(col, 0),  4),
            'group'  : ('risk' if col.startswith('risk_') else
                        'interaction' if col.startswith('interact_') else
                        'ratio' if col.startswith('feat_') else
                        'zscore' if col.startswith('zscore_') else
                        'flag' if col.startswith('flag_') else
                        'bin' if col.startswith('bin_') else 'baseline'),
        })
    return {'feature_count': len(feature_cols), 'features': feat_info}


print('✅ FastAPI application defined with 5 endpoints:')
print('   GET  /health')
print('   POST /predict')
print('   POST /scan-document')
print('   POST /batch-predict')
print('   GET  /features')

## Cell 8 — Launch API with ngrok Public URL

In [ ]:
from pyngrok import ngrok, conf

# ── Option A: No auth token (limited — works for demo) ─────────────────────────
# If you have an ngrok account, set your auth token below for unlimited tunnels
# ngrok.set_auth_token('YOUR_NGROK_TOKEN_HERE')

API_PORT = 8000

# Start ngrok tunnel
ngrok.kill()   # Kill any existing tunnels
tunnel  = ngrok.connect(API_PORT, 'http')
API_URL = tunnel.public_url

print(f'\n🌐 ngrok tunnel active:')
print(f'   Public API URL : {API_URL}')
print(f'   Swagger UI     : {API_URL}/docs')
print(f'   Health check   : {API_URL}/health')

# Start uvicorn in background thread
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=API_PORT, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)   # Give server time to start

# Quick health check
try:
    resp = requests.get(f'http://localhost:{API_PORT}/health', timeout=5)
    if resp.status_code == 200:
        health = resp.json()
        print(f'\n✅ API is running!')
        print(f'   Status    : {health["status"]}')
        print(f'   Model AUC : {health["model_auc"]}')
        print(f'   Features  : {health["feature_count"]}')
    else:
        print(f'⚠️  API returned status {resp.status_code}')
except Exception as e:
    print(f'⚠️  Could not reach API: {e}')

## Cell 9 — Test All API Endpoints

In [ ]:
BASE = f'http://localhost:{API_PORT}'

print('🧪 Testing all API endpoints...\n')

# ── Test 1: Health ─────────────────────────────────────────────────────────────
r = requests.get(f'{BASE}/health')
print(f'[1] GET /health → {r.status_code}')
print(f'    {r.json()}\n')

# ── Test 2: Predict (HIGH FRAUD values) ────────────────────────────────────────
high_fraud_payload = {
    'features': {
        'ip_avg_reimbursement'    : 35000.0,   # Very high
        'ip_avg_stay_days'        : 45.0,      # Very long stay
        'ip_claim_count'          : 120.0,     # High volume
        'total_unique_patients'   : 80.0,      # Many patients
        'ip_avg_chronic_cond'     : 6.5,       # Many diagnoses
        'ip_unique_attending_phys': 35.0,      # Many physicians
    },
    'explain': True
}
r = requests.post(f'{BASE}/predict', json=high_fraud_payload)
resp = r.json()
print(f'[2] POST /predict (HIGH FRAUD) → {r.status_code}')
print(f'    fraud_score : {resp.get("fraud_score")}')
print(f'    risk_tier   : {resp.get("risk_tier")}')
print(f'    decision    : {"FRAUD" if resp.get("fraud_predicted") else "LEGIT"}')
print(f'    top reason  : {resp.get("top_reasons",[{}])[0].get("text","N/A")}')
print()

# ── Test 3: Predict (NORMAL values) ───────────────────────────────────────────
normal_payload = {
    'features': {
        'ip_avg_reimbursement' : 4200.0,
        'ip_avg_stay_days'     : 3.2,
        'ip_claim_count'       : 12.0,
        'total_unique_patients': 9.0,
        'ip_avg_chronic_cond'  : 1.8,
    },
    'explain': True
}
r = requests.post(f'{BASE}/predict', json=normal_payload)
resp = r.json()
print(f'[3] POST /predict (NORMAL) → {r.status_code}')
print(f'    fraud_score : {resp.get("fraud_score")}')
print(f'    risk_tier   : {resp.get("risk_tier")}')
print(f'    decision    : {"FRAUD" if resp.get("fraud_predicted") else "LEGIT"}')
print()

# ── Test 4: Batch predict ──────────────────────────────────────────────────────
batch_payload = [
    {'features': {'ip_avg_reimbursement': 35000.0, 'ip_avg_stay_days': 45.0}, 'explain': False},
    {'features': {'ip_avg_reimbursement': 4200.0,  'ip_avg_stay_days': 3.2},  'explain': False},
    {'features': {'ip_avg_reimbursement': 18000.0, 'ip_avg_stay_days': 22.0}, 'explain': False},
]
r = requests.post(f'{BASE}/batch-predict', json=batch_payload)
resp = r.json()
print(f'[4] POST /batch-predict → {r.status_code}')
print(f'    Total scored : {resp.get("total")}')
print(f'    Fraud flagged: {resp.get("fraud_count")}')
for i, res in enumerate(resp.get('results',[]), 1):
    print(f'    Provider {i}: score={res.get("fraud_score")}  tier={res.get("risk_tier")}')
print()

# ── Test 5: Scan document ──────────────────────────────────────────────────────
test_pdf = '/content/test_claim_HIGH_FRAUD.pdf'
if os.path.exists(test_pdf):
    with open(test_pdf, 'rb') as f:
        r = requests.post(f'{BASE}/scan-document',
                          files={'file': ('test_claim.pdf', f, 'application/pdf')})
    resp = r.json()
    print(f'[5] POST /scan-document → {r.status_code}')
    print(f'    fraud_score  : {resp.get("fraud_score")}')
    print(f'    risk_tier    : {resp.get("risk_tier")}')
    print(f'    OCR regions  : {resp.get("ocr_regions")}')
    print(f'    Fields found : {list(resp.get("extracted_fields",{}).keys())}')
else:
    print(f'[5] POST /scan-document → SKIPPED (run M6 Cell 11 first to create test PDFs)')

print(f'\n✅ All endpoints tested.')
print(f'   Full Swagger docs: {API_URL}/docs')

## Cell 10 — Full Gradio Dashboard (4 Tabs)

In [ ]:
# ── Gradio helper functions ────────────────────────────────────────────────────
def gradio_scan_doc(file_obj):
    if file_obj is None:
        return None, '⚠️ Upload a file first', ''
    try:
        with open(file_obj.name,'rb') as f:
            contents = f.read()
        result, pil_img, ocr_results = scan_bytes(contents, file_obj.name)

        # Annotate image
        draw = ImageDraw.Draw(pil_img.copy())
        annotated = pil_img.copy()

        score  = result['fraud_score']
        tier   = result['risk_tier']
        pred   = result['fraud_predicted']
        emojis = {'CRITICAL':'🚨','HIGH':'⚠️','MEDIUM':'🟡','LOW':'✅'}

        score_txt = (
            f"{emojis.get(tier,'❓')} FRAUD SCORE: {score:.4f}\n"
            f"Risk Tier  : {tier}\n"
            f"Decision   : {'🚨 FRAUD' if pred else '✅ LEGITIMATE'}\n"
            f"Action     : {result['action']}\n"
            f"OCR regions: {result['ocr_regions']}"
        )
        reasons_txt = 'TOP FRAUD SIGNALS:\n\n'
        for i,r in enumerate(result['top_reasons'][:5],1):
            reasons_txt += f"{i}. {r['text']}\n   SHAP: +{r['shap']:.4f}\n\n"
        reasons_txt += 'EXTRACTED FIELDS:\n'
        for k,v in result['extracted_fields'].items():
            reasons_txt += f"  {k}: {v}\n"
        return annotated, score_txt, reasons_txt
    except Exception as e:
        return None, f'❌ Error: {e}', ''


def gradio_manual_predict(
    avg_reimb, stay_days, claim_count,
    unique_patients, chronic_cond, unique_phys
):
    try:
        feat = {
            'ip_avg_reimbursement'    : float(avg_reimb),
            'ip_avg_stay_days'        : float(stay_days),
            'ip_claim_count'          : float(claim_count),
            'total_unique_patients'   : float(unique_patients),
            'ip_avg_chronic_cond'     : float(chronic_cond),
            'ip_unique_attending_phys': float(unique_phys),
        }
        feat_df = feat_df_from_dict(feat)
        preds   = ensemble_predict(feat_df)
        tier, action = get_risk_tier(preds['fraud_score'])
        reasons = get_shap_reasons(feat_df)
        emojis = {'CRITICAL':'🚨','HIGH':'⚠️','MEDIUM':'🟡','LOW':'✅'}
        out = (
            f"{emojis.get(tier,'❓')} FRAUD SCORE: {preds['fraud_score']:.4f}\n"
            f"Risk Tier  : {tier}\n"
            f"Decision   : {'🚨 FRAUD' if preds['fraud_score']>=THRESHOLD else '✅ LEGITIMATE'}\n"
            f"Action     : {action}\n\n"
            f"Model Scores:\n"
            f"  XGBoost  : {preds['xgb_score']:.4f}\n"
            f"  LightGBM : {preds['lgbm_score']:.4f}\n"
            f"  RF       : {preds['rf_score']:.4f}\n\n"
            f"Top Fraud Signals:\n"
        )
        for i,r in enumerate(reasons[:5],1):
            out += f"  {i}. {r['text']}  [SHAP: +{r['shap']:.4f}]\n"
        return out
    except Exception as e:
        return f'❌ Error: {e}'


def gradio_batch_csv(file_obj):
    if file_obj is None:
        return None, '⚠️ Upload a CSV file'
    try:
        df = pd.read_csv(file_obj.name)
        results = []
        for _, row in df.iterrows():
            feat = {c: float(v) for c,v in row.items()
                    if c in feature_cols and pd.notna(v)}
            feat_df = feat_df_from_dict(feat)
            preds   = ensemble_predict(feat_df)
            tier, _ = get_risk_tier(preds['fraud_score'])
            results.append({
                **{c: row.get(c,'') for c in df.columns},
                'fraud_score'    : preds['fraud_score'],
                'risk_tier'      : tier,
                'fraud_predicted': preds['fraud_score'] >= THRESHOLD,
            })
        out_df   = pd.DataFrame(results)
        out_path = f'/tmp/batch_results_{datetime.now().strftime("%H%M%S")}.csv'
        out_df.to_csv(out_path, index=False)
        fraud_ct = out_df['fraud_predicted'].sum()
        summary  = (
            f'✅ Scored {len(out_df)} providers\n'
            f'Flagged as fraud: {fraud_ct} ({fraud_ct/len(out_df)*100:.1f}%)\n'
            f'Results saved to: {out_path}'
        )
        return out_df[['fraud_score','risk_tier','fraud_predicted']].head(20), summary
    except Exception as e:
        return None, f'❌ Error: {e}'


# ── Build 4-tab Gradio interface ───────────────────────────────────────────────
with gr.Blocks(
    title = 'Insurance Fraud Detection System',
    theme = gr.themes.Base(primary_hue='teal', neutral_hue='slate'),
) as dashboard:

    gr.Markdown("""
    # 🛡️ Insurance Fraud Detection System
    **Multimodal AI | Document Intelligence | Explainability**
    Model: XGBoost + LightGBM + Random Forest Ensemble | AUC-ROC: {auc}
    """.format(auc=metadata['val_auc_roc']))

    # ── Tab 1: Document Scanner ────────────────────────────────────────────────
    with gr.Tab('📄 Document Scanner'):
        gr.Markdown('Upload a claim PDF or image → OCR → AI fraud scoring → highlighted report')
        with gr.Row():
            with gr.Column(scale=1):
                doc_file = gr.File(label='Upload Claim Document',
                                   file_types=['.pdf','.jpg','.jpeg','.png'])
                doc_btn  = gr.Button('🔍 Scan Document', variant='primary')
            with gr.Column(scale=2):
                doc_img  = gr.Image(label='Annotated Document')
        with gr.Row():
            doc_score   = gr.Textbox(label='Fraud Score & Decision', lines=5)
            doc_reasons = gr.Textbox(label='Fraud Signals & Extracted Fields', lines=8)
        doc_btn.click(gradio_scan_doc, [doc_file],
                      [doc_img, doc_score, doc_reasons])

    # ── Tab 2: Manual Feature Input ────────────────────────────────────────────
    with gr.Tab('⚙️ Manual Feature Input'):
        gr.Markdown('Enter provider-level features manually to get an instant fraud score.')
        with gr.Row():
            avg_reimb   = gr.Number(label='Avg IP Reimbursement ($)',   value=5000)
            stay_days   = gr.Number(label='Avg Hospital Stay (days)',    value=5)
            claim_count = gr.Number(label='Inpatient Claim Count',       value=20)
        with gr.Row():
            unique_pts  = gr.Number(label='Total Unique Patients',       value=15)
            chronic_cond= gr.Number(label='Avg Chronic Conditions',      value=2.5)
            unique_phys = gr.Number(label='Unique Attending Physicians', value=8)
        predict_btn = gr.Button('🤖 Predict Fraud Score', variant='primary')
        manual_out  = gr.Textbox(label='Fraud Prediction Result', lines=12)
        predict_btn.click(
            gradio_manual_predict,
            [avg_reimb, stay_days, claim_count, unique_pts, chronic_cond, unique_phys],
            [manual_out]
        )
        gr.Markdown('**Typical fraud indicators:** Reimbursement > $15,000 | Stay > 20 days | Patients > 50')

    # ── Tab 3: Batch CSV Scoring ───────────────────────────────────────────────
    with gr.Tab('📊 Batch CSV Scoring'):
        gr.Markdown(
            'Upload a CSV where each row is a provider. '
            'Columns should match feature names (see /features endpoint). '
            'Any missing columns default to population mean.'
        )
        csv_file  = gr.File(label='Upload Provider CSV', file_types=['.csv'])
        batch_btn = gr.Button('📈 Score All Providers', variant='primary')
        batch_tbl = gr.Dataframe(label='Results Preview (top 20 rows)')
        batch_sum = gr.Textbox(label='Batch Summary', lines=4)
        batch_btn.click(gradio_batch_csv, [csv_file], [batch_tbl, batch_sum])

    # ── Tab 4: API Info ────────────────────────────────────────────────────────
    with gr.Tab('🔗 API Reference'):
        gr.Markdown(f"""
        ## Live API Endpoints

        | Endpoint | Method | Description |
        |---|---|---|
        | `/health` | GET | API status + model info |
        | `/predict` | POST | Score provider from features |
        | `/scan-document` | POST | Upload PDF/image → OCR → fraud score |
        | `/batch-predict` | POST | Score multiple providers at once |
        | `/features` | GET | List all feature names + population stats |
        | `/docs` | GET | Interactive Swagger UI |

        ## Example: POST /predict
        ```json
        {{
          "features": {{
            "ip_avg_reimbursement": 15000,
            "ip_avg_stay_days": 25,
            "ip_claim_count": 80
          }},
          "explain": true
        }}
        ```

        **Public API URL:** `{API_URL}`
        **Swagger Docs:** [{API_URL}/docs]({API_URL}/docs)
        """
        )

print('✅ Gradio dashboard built (4 tabs).')

## Cell 11 — Launch Gradio Dashboard

In [ ]:
print('🚀 Launching Gradio dashboard...')
print('   (share=True gives you a public URL valid for 72 hours)')

dashboard.launch(
    share       = True,
    debug       = False,
    server_port = 7860,
    show_error  = True,
)

# Both the FastAPI (port 8000) and Gradio (port 7860) are now running
print(f'\n✅ FULL SYSTEM LIVE:')
print(f'   FastAPI + Swagger : {API_URL}/docs')
print(f'   Gradio Dashboard  : (see link above from Gradio)')
print(f'\n   Share either URL with your faculty for the live demo!')

## Cell 12 — Project Complete Summary

In [ ]:
print('█'*65)
print('  🎉 INSURANCE FRAUD DETECTION SYSTEM — COMPLETE')
print('█'*65)

modules = [
    ('M1', 'Data Merging & Preprocessing',
     'Merged 8 CSVs → provider-level dataset → SMOTE balancing'),
    ('M2', 'EDA & Visualization',
     '8 publication-quality plots → key fraud signals identified'),
    ('M3', 'Feature Engineering',
     '57 features across 7 groups → MI-ranked → parquet output'),
    ('M4', 'Model Training',
     f'Ensemble AUC={metadata["val_auc_roc"]} | F1={metadata["val_f1"]} | Optuna-tuned'),
    ('M5', 'XAI Explainability',
     'SHAP waterfall + NL reasons + per-provider PDF report'),
    ('M6', 'Document Scanner',
     'OCR → field parsing → per-field suspicion → highlighted image'),
    ('M7', 'Deployment',
     'FastAPI (5 endpoints) + Gradio (4 tabs) + ngrok public URL'),
]

print()
for code, name, desc in modules:
    print(f'  ✅ {code}: {name}')
    print(f'     {desc}')

print(f'\n  📊 Model Performance (Validation Set):')
print(f'     AUC-ROC   : {metadata["val_auc_roc"]}')
print(f'     F1-Score  : {metadata["val_f1"]}')
print(f'     Precision : {metadata["val_precision"]}')
print(f'     Recall    : {metadata["val_recall"]}')

print(f'\n  🌐 Live Demo:')
print(f'     FastAPI Swagger : {API_URL}/docs')
print(f'     Gradio UI       : (public link from Cell 11)')

print(f'\n  📁 Project Files:')
print(f'     7 Colab notebooks  → complete pipeline')
print(f'     5 trained models   → {MODEL_DIR}')
print(f'     8 EDA/SHAP plots   → {PLOT_DIR if "PLOT_DIR" in dir() else "plots/ folder"}')
print(f'     PDF reports        → {REPORT_DIR}')
print(f'     Scanned documents  → {SCAN_DIR}')
print('█'*65)